# 31 孟加拉国 DIA 特征级风险整合与优先级排序（V2）

回连 MS-DIAL 样品峰高，整合五个冻结模型端点、预测不确定性、检出频率和丰度，并根据加合物中性质量与 RT 对 Pos/Neg 特征进行跨极性聚类。

> 结果是未知特征的探索性优先级，不是化学鉴定，也不是法规意义上的 PBT 判定。

内核：`env_ms`。

In [ ]:
from pathlib import Path
import sys,json
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'pbt_msms').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'pbt_msms').exists(),'找不到项目根目录'
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
print('ROOT =',ROOT); print('Python =',sys.executable)


In [ ]:
from pbt_msms.bangladesh_dia_external import run_ranking,paths
manifest=run_ranking(ROOT)
print(json.dumps(manifest,ensure_ascii=False,indent=2))
assert manifest['status']=='FEATURE_PRIORITIZATION_FROZEN'


In [ ]:
import pandas as pd,numpy as np
p=paths(ROOT); OUT=p['case']/'06_revealed_evaluation'/'31_feature_prioritization_v2'
features=pd.read_csv(OUT/'31_feature_priority_ranking.csv.gz')
clusters=pd.read_csv(OUT/'31_cross_polarity_cluster_ranking.csv.gz')
sensitivity=pd.read_csv(OUT/'31_ranking_sensitivity.csv')
display(features.groupby('mode').agg(n=('spectrum_id','size'),all5_retained=('all_five_endpoints_retained','sum'),median_priority=('priority_score_primary','median')))
display(sensitivity)
display(clusters.head(20)[['cluster_rank','cross_polarity_cluster','modes','member_spectrum_ids','neutral_mass','rt_min','priority_score_primary','retained_endpoint_n']])


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
features.priority_score_primary.hist(bins=60,ax=axes[0]); axes[0].set_title('Priority score')
axes[1].scatter(features.exposure_component,features.hazard_component,c=features.priority_score_primary,s=5,alpha=.45,cmap='viridis'); axes[1].set(xlabel='Exposure component',ylabel='Hazard component',title='Hazard–exposure map')
features.groupby('mode').retained_endpoint_n.value_counts().unstack(fill_value=0).plot.bar(ax=axes[2]); axes[2].set_title('Retained endpoints by mode'); axes[2].set_ylabel('Features')
plt.tight_layout(); plt.show()


## 后续使用原则

优先使用 `31_top100_clusters.csv` 进入结构注释，避免同一中性物质在正负模式重复占位。结构注释前不要把 `spectrum_id` 或 `Alignment ID` 表述为已鉴定污染物。